# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas72O5/flyrank-ml-internship_week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = One unique content item (content_hash_id) for a specific client (client_hash_id).
Window: Features are derived from the mid-panel month of March 2026 (month=2026-03).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import pyarrow.parquet as pq
import requests
from io import BytesIO
from google.colab import userdata

# 1. Get Token and Setup Headers
token = userdata.get('HF_TOKEN')
headers = {"Authorization": f"Bearer {token}"}

# 2. URL for a SINGLE parquet file in March (easier to load for verification)
# We pick part '0' of the month
url = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/part.0.parquet"

# 3. Download the data into memory
response = requests.get(url, headers=headers)
if response.status_code == 200:
    # Read the parquet data
    table = pq.read_table(BytesIO(response.content))
    df_march = table.to_pandas()

    print("Success! Data loaded via Pandas Fallback.")

    # Verify the Grain (Fact 1)
    grain = df_march.groupby(['client_hash_id', 'content_hash_id']).size().reset_index(name='days_count').head(5)
    print("\nFact 1: The Grain")
    display(grain)

    # Verify the Date Span (Fact 2)
    print(f"\nFact 2: Date Span: {df_march['report_date'].min()} to {df_march['report_date'].max()}")

else:
    print(f"Error: Could not access data. Status Code: {response.status_code}")
    print("Check if you have accepted the gated access terms on Hugging Face.")

Error: Could not access data. Status Code: 404
Check if you have accepted the gated access terms on Hugging Face.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Features (Clues): content_age_days, avg_position, ctr, sessions_90d, word_count. (Available at decision time).
Label (Target): is_decaying_champion. (A proxy created from trend_direction == 'down' where avg_position < 20).
Context: client_hash_id, content_hash_id. (Used for grouping and client-holdout).
Excluded: trend_pct and health_score.
Why: trend_pct is the exact number used to calculate the label (Leakage). health_score is a product-calculated decision; using it would be circular reasoning.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# CODE CELL for Section 2
# Verify that our features exist in the table
feature_verify = con.execute(f"""
    SELECT content_age_days, avg_position, ctr, sessions_90d, word_count
    FROM read_parquet('{path}')
    LIMIT 1
""").df()
print("Features confirmed available in warehouse:")
display(feature_verify)

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/*.parquet' (HTTP 401)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Row Counts: How many pages are we actually analyzing in this slice?
counts = con.execute(f"SELECT count(distinct content_hash_id) as total_pages FROM read_parquet('{path}')").df()

# 2. Availability Check: GSC and GA4 data must be TRUE
availability = con.execute(f"""
    SELECT
        count(*) as total_rows,
        count(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_ok,
        count(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_ok
    FROM read_parquet('{path}')
""").df()

# 3. Target Snapshot: How many 'Champions' are actually in a down trend in March?
target_snapshot = con.execute(f"""
    SELECT count(*) as decaying_champions
    FROM read_parquet('{path}')
    WHERE trend_direction = 'down' AND avg_position < 20
""").df()

print(f"Total Unique Pages: {counts.iloc[0,0]}")
print("\nData Availability (GSC vs GA4):")
display(availability)
print(f"\nPotential 'Decaying Champion' targets in this month: {target_snapshot.iloc[0,0]}")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. Unbalanced History: As noted in the dim_clients table, some clients have 12+ months of history, while others have only 3. This may introduce bias toward more established websites.
2. Tracking Gaps: Rows from before a client's ga4_data_start contain search data only. I must use ga4_data_available IS TRUE as a filter if I want to use engagement metrics.
3. Observational Only: The data shows what happened, but not why. It cannot distinguish between a content problem and a sudden competitor bid increase.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Prove the 'Unbalanced History' limitation
client_history = con.execute(f"""
    SELECT client_hash_id, count(distinct report_date) as history_days
    FROM read_parquet('{path}')
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 5
""").df()
print("Evidence of Unbalanced history (Top 5 clients by data density in March):")
display(client_history)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.